# Simulation for the probability of sites in the clonal frame

In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
def aff_P1(delta, L):
    return delta / (delta + L - 1)


def aff_P21(delta, L):
    return (1 + delta * (1 - (1 - 1/delta)**(L-1))) / L


def aff_P22(delta, L):
    return (delta * (1 - (1 - 1/delta)**(L))) / L


def CF_P(N, rho, aff_P):
    return math.gamma(rho * aff_P + 1) * math.gamma(N) / math.gamma(rho * aff_P + N)

In [3]:
def birth_death_recomb_sim_whole1(N, rho, delta, L):
    """
    Simulate recombination events based on a birth-death process

    Simulate recombinations to compute the probability of a single site in a clona frame.

    param:
    N: A single integer for initial value - the number of lineages.
    rho: The birth rate - the recombination rate.
    delta: The mean value of the recombine segment.
    L: The number of sites.
    return:
    Hitting time, number of recombinations, number of final lineages,
    proportion of sites that are in the clonal frame.
    """
    if not isinstance(N, (int, np.integer)):
        raise ValueError(f"`N` must be a single integer!")
    if not isinstance(L, (int, np.integer)):
        raise ValueError(f"`L` must be a single integer!")

    t = 0.0
    k = N
    clonal_k = N
    non_clonal_k = 0
    recomb_num = 0
    in_CF = np.ones(L, dtype=bool)
    
    while clonal_k > 1:
        rate = k * (k - 1 + rho) / 2
        t += np.random.exponential(scale=1/rate)

        if np.random.random() < (k - 1) / (k - 1 + rho): # coalescence event
            prob_two_clonal = clonal_k * (clonal_k - 1) / 2
            prob_one_clonal = clonal_k * non_clonal_k
            prob_two_non_clonal = non_clonal_k * (non_clonal_k - 1) / 2
            prob = np.array([prob_two_clonal, prob_one_clonal, prob_two_non_clonal])
            choice = np.random.choice(3, p=prob / np.sum(prob))
            if choice == 0:
                clonal_k -= 1
            elif choice == 1:
                non_clonal_k -= 1
            else:
                non_clonal_k -= 1
            k -= 1

        else:                                            # recombination event
            recomb_num += 1
            start_pos = np.random.randint(0, L)
            recomb_length = np.random.geometric(p=1/delta)
            end_pos = start_pos + recomb_length
            end_pos = end_pos % L

            prob_on_clonal = clonal_k / k
            if np.random.random() < prob_on_clonal:
                if recomb_length >= L: # All sites are from donor, so no sites are in the clonal frame.
                    in_CF[:] = False
                else:
                    not_in_recomb = np.ones(L, dtype=bool)
                    if start_pos < end_pos:
                        not_in_recomb[start_pos:(end_pos+1)] = False
                    else:
                        not_in_recomb[start_pos:]   = False
                        not_in_recomb[:(end_pos+1)] = False
                    in_CF = np.logical_and(in_CF, not_in_recomb)

            non_clonal_k += 1
            k += 1

    return [t, recomb_num, k, clonal_k, np.mean(in_CF)]

In [4]:
def birth_death_recomb_sim_whole2(N, rho, delta, L):
    """
    Simulate recombination events based on a birth-death process

    Simulate recombinations to compute the probability of a single site in a clona frame.

    param:
    N: A single integer for initial value - the number of lineages.
    rho: The birth rate - the recombination rate.
    delta: The mean value of the recombine segment.
    L: The number of sites.
    return:
    Hitting time, number of recombinations, number of final lineages,
    proportion of sites that are in the clonal frame.
    """
    if not isinstance(N, (int, np.integer)):
        raise ValueError(f"`N` must be a single integer!")
    if not isinstance(L, (int, np.integer)):
        raise ValueError(f"`L` must be a single integer!")

    t = 0.0
    k = N
    clonal_k = N
    non_clonal_k = 0
    recomb_num = 0
    in_CF = np.ones(L, dtype=bool)
    
    while clonal_k > 1:
        rate = k * (k - 1 + rho) / 2
        t += np.random.exponential(scale=1/rate)

        if np.random.random() < (k - 1) / (k - 1 + rho): # coalescence event
            prob_two_clonal = clonal_k * (clonal_k - 1) / 2
            prob_one_clonal = clonal_k * non_clonal_k
            prob_two_non_clonal = non_clonal_k * (non_clonal_k - 1) / 2
            prob = np.array([prob_two_clonal, prob_one_clonal, prob_two_non_clonal])
            choice = np.random.choice(3, p=prob / np.sum(prob))
            if choice == 0:
                clonal_k -= 1
            elif choice == 1:
                non_clonal_k -= 1
            else:
                non_clonal_k -= 1
            k -= 1

        else:                                            # recombination event
            recomb_num += 1
            start_pos = np.random.randint(0, L)
            recomb_length = np.random.geometric(p=1/delta)
            end_pos = start_pos + recomb_length - 1
            end_pos = end_pos % L

            prob_on_clonal = clonal_k / k
            if np.random.random() < prob_on_clonal:
                if recomb_length >= L: # All sites are from donor, so no sites are in the clonal frame.
                    in_CF[:] = False
                else:
                    not_in_recomb = np.ones(L, dtype=bool)
                    if start_pos <= end_pos:
                        not_in_recomb[start_pos:(end_pos+1)] = False
                    else:
                        not_in_recomb[start_pos:]   = False
                        not_in_recomb[:(end_pos+1)] = False
                    in_CF = np.logical_and(in_CF, not_in_recomb)

            non_clonal_k += 1
            k += 1

    return [t, recomb_num, k, clonal_k, np.mean(in_CF)]

In [5]:
def end_pos_prob(start_pos, L, delta):
    prob = np.zeros(L)
    for s in range(start_pos, L):
        if s < L - 1:
            prob[s] = (1 - 1/delta)**(s - start_pos) / delta
        else:
            prob[s] = (1 - 1/delta)**(s - start_pos + 1)
    return prob / np.sum(prob)


def birth_death_recomb_sim_seg(N, rho, delta, L):
    """
    Simulate recombination events based on a birth-death process

    Simulate recombinations to compute the probability of a single site in a clona frame.

    param:
    N: A single integer for initial value - the number of lineages.
    rho: The birth rate - the recombination rate.
    delta: The mean value of the recombine segment.
    L: The number of sites.
    return:
    Hitting time, number of recombinations,
    number of final lineages,
    number of clonal lineages,
    and the proportion of sites that are in the clonal frame.
    """
    if not isinstance(N, (int, np.integer)):
        raise ValueError(f"`N` must be a single integer!")
    if not isinstance(L, (int, np.integer)):
        raise ValueError(f"`L` must be a single integer!")

    t = 0.0
    k = N
    clonal_k = N
    non_clonal_k = 0
    recomb_num = 0
    in_CF = np.ones(L, dtype=bool)

    probstart = np.ones(L, dtype=float)
    probstart[0] = delta
    probstart = probstart / np.sum(probstart)
    
    while clonal_k > 1:
        rate = k * (k - 1 + rho) / 2
        t += np.random.exponential(scale=1/rate)

        if np.random.random() < (k - 1) / (k - 1 + rho): # coalescence event
            prob_two_clonal = clonal_k * (clonal_k - 1) / 2
            prob_one_clonal = clonal_k * non_clonal_k
            prob_two_non_clonal = non_clonal_k * (non_clonal_k - 1) / 2
            prob = np.array([prob_two_clonal, prob_one_clonal, prob_two_non_clonal])
            choice = np.random.choice(3, p=prob / np.sum(prob))
            if choice == 0:
                clonal_k -= 1
            elif choice == 1:
                non_clonal_k -= 1
            else:
                non_clonal_k -= 1
            k -= 1

        else:                                            # recombination event
            recomb_num += 1
            start_pos = np.random.choice(L, p=probstart)
            end_pos = np.random.choice(L, p=end_pos_prob(start_pos, L, delta))

            prob_on_clonal = clonal_k / k
            if np.random.random() < prob_on_clonal:
                not_in_recomb = np.ones(L, dtype=bool)
                not_in_recomb[start_pos:(end_pos+1)] = False
                in_CF = np.logical_and(in_CF, not_in_recomb)

            non_clonal_k += 1
            k += 1

    return [t, recomb_num, k, clonal_k, np.mean(in_CF)]

In [6]:
N = 10
rho = 1
delta = 5
L = 20

P1 = aff_P1(delta, L)
P21 = aff_P21(delta, L)
P22 = aff_P22(delta, L)
CF_P1 = CF_P(N, rho, P1)
CF_P21 = CF_P(N, rho, P21)
CF_P22 = CF_P(N, rho, P22)

print(f"aff_P1: {P1}, aff_P21: {P21}, aff_P22: {P22}")
print(f"CF_P1: {CF_P1}, CF_P21: {CF_P21}, CF_P22: {CF_P22}")

aff_P1: 0.20833333333333334, aff_P21: 0.2963971202981036, aff_P22: 0.24711769623848284
CF_P1: 0.5717120651070936, CF_P21: 0.45861549458602974, CF_P22: 0.5182783388007098


In [7]:
list_sim_results_whole1 = []
list_sim_results_whole2 = []
list_sim_results_seq = []
for _ in tqdm(range(100_000)):
    sim_result_whole = birth_death_recomb_sim_whole1(N, rho, delta, L)
    sim_result_whole2 = birth_death_recomb_sim_whole2(N, rho, delta, L)
    sim_result_seq = birth_death_recomb_sim_seg(N, rho, delta, L)
    list_sim_results_whole1.append(sim_result_whole)
    list_sim_results_whole2.append(sim_result_whole2)
    list_sim_results_seq.append(sim_result_seq)

100%|██████████| 100000/100000 [00:57<00:00, 1726.79it/s]


In [8]:
array_sim_results_whole1 = np.array(list_sim_results_whole1)
array_sim_results_whole2 = np.array(list_sim_results_whole2)
array_sim_results_seq = np.array(list_sim_results_seq)
array_sim_results_whole1.shape, array_sim_results_whole2.shape, array_sim_results_seq.shape

((100000, 5), (100000, 5), (100000, 5))

In [9]:
np.mean(array_sim_results_seq[:, 4]), np.mean(array_sim_results_whole1[:, 4]), np.mean(array_sim_results_whole2[:, 4])

(np.float64(0.5791505), np.float64(0.4574695), np.float64(0.5181905))